# NLP semántico en español: embeddings, cosine similarity y BERTopic

Objetivo: usar **embeddings semánticos** para representar textos en español, medir **similitud coseno**, detectar temas con **BERTopic** y calcular un **coherence score**.

La idea es:
1. Ejecutar primero este ejemplo.
2. Entender qué hace cada bloque.
3. Sustituir después el dataframe por los datos de tu trabajo.

## 0. Instalación de librerías

In [6]:
!pip install -q sentence-transformers bertopic gensim pandas scikit-learn umap hdbscan

ERROR: Could not find a version that satisfies the requirement umap (from versions: none)
ERROR: No matching distribution found for umap


## 1. Imports

In [7]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from bertopic import BERTopic

from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

pd.set_option("display.max_colwidth", 200)

## 2. Dataset de ejemplo en español

El dataset mezcla textos sobre universidad, tecnología, salud y vivienda. Hay temas bastante claros, pero también documentos con zonas de solapamiento: tecnología en universidades, salud mental en estudiantes, vivienda para jóvenes, etc.

In [8]:
df = pd.read_csv('datos_clean.csv')

display(df)

,periodico,titulo,texto,url,texto_clean
0,eldiario.es,La resaca de El Alcoraz,"En la competición de visualizaciones del vídeo más impactante del Día de Aragón, en esta ocasión no ganó ni el del Gobierno de Aragón ni el de la empresa cervecera de la tierra. Ganaron contundent...",https://www.eldiario.es/aragon/el-prismatico/resaca-alcoraz_132_13188026.html,competición visualizaciones vídeo impactante día aragón ocasión ganó gobierno aragón empresa cervecera tierra ganaron contundentemente deplorables imágenes alcoraz proyectando mundo pocas horas de...
1,eldiario.es,Juicio a Vivotecnia: cómo demostrar un exceso de sufrimiento en actividades que ya admiten un gran sufrimiento legal,"Hay asuntos que una no abandona al salir del juzgado. Casos que se quedan adheridos a la piel, que modifican la manera de mirar el mundo y que convierten determinados detalles un sonido, una pala...",https://www.eldiario.es/caballodenietzsche/juicio-vivotecnia-maltrato-animal-ciencia_132_13187982.html,asuntos abandona salir juzgado casos quedan adheridos piel modifican manera mirar mundo convierten determinados detalles sonido palabra número imposible olvidar abogada aprendido procedimientos es...
2,eldiario.es,La resaca de El Alcoraz,"En la competición de visualizaciones del vídeo más impactante del Día de Aragón, en esta ocasión no ganó ni el del Gobierno de Aragón ni el de la empresa cervecera de la tierra. Ganaron contundent...",https://www.eldiario.es/aragon/el-prismatico/resaca-alcoraz_132_13188029.html,competición visualizaciones vídeo impactante día aragón ocasión ganó gobierno aragón empresa cervecera tierra ganaron contundentemente deplorables imágenes alcoraz proyectando mundo pocas horas de...
3,eldiario.es,Mentiras y cintas de vídeo,"Como un gélido bisturí explorador de las miserias humanas definió Miguel Ángel Palomo la película 'Sexo, mentiras y cintas de vídeo', la aclamada y ochentera ópera prima de Steven Soderbergh . E...",https://www.eldiario.es/aragon/el-prismatico/mentiras-cintas-video_132_13185168.html,gélido bisturí explorador miserias humanas definió miguel ángel palomo película sexo mentiras cintas vídeo aclamada ochentera ópera prima steven soderbergh semana aragón investidura jorge azcón pr...
4,eldiario.es,Vosotros sois la televisión,Un minuto y diez segundos. Eso es lo que dura un vídeo de informativos en televisión. La duración estándar. En ese tiempo quienes trabajamos en ese medio hemos de ser capaces de contar una vida; e...,https://www.eldiario.es/aragon/el-prismatico/television_132_13184101.html,minuto diez segundos dura vídeo informativos televisión duración estándar tiempo trabajamos medio ser capaces contar vida explicar complejidad universo entero mostrarles imagen inmensidad detalle ...
...,...,...,...,...,...
9995,el español,a vox en grito: el radicalismo de sánchez dispara a la extrema derecha,"abascal y sánchez, en una sesión de control. el sondeo de sociométrica que hoy publica el español dibuja un escenario de emergencia para el bloque gubernamental. el psoe, siguiendo la inercia de l...",https://www.elespanol.com/opinion/editoriales/20260216/vox-grito-radicalismo-sanchez-dispara-extrema-derecha/1003744131379_14.html,abascal sánchez sesión control sondeo sociométrica hoy publica español dibuja escenario emergencia bloque gubernamental psoe siguiendo inercia recientes descalabros extremadura aragónigualaría ele...
9996,el español,el dilema de ciudadanos,"los argumentos del actual portavoz paradisputar el liderazgo a la mujera la que, hace sólo tres meses, rendía exagerada pleitesía como su “más leal escudero” son tan inconsistentes que merecen poc...",https://www.elespanol.com/opinion/carta-del-director/20221211/dilema-ciudadanos/725127481_20.html,argumentos actual portavoz paradisputar liderazgo mujera hace sólo tres meses rendía exagerada pleitesía leal escudero tan inconsistentes merecen comentario dilema ciudadanos consiste decidir entr...
9997,el español,dos líderes de izquierdas dinamitan la sol

## 3. Configuración

In [9]:
TEXT_COLUMN = "texto"
GROUP_COLUMN = "periodico"

documents = df[TEXT_COLUMN].dropna().astype(str).tolist()

print(f"Número de documentos: {len(documents)}")

Número de documentos: 10000


## 4. Embeddings semánticos

Usamos un modelo multilingüe de `sentence-transformers`, adecuado para textos en español.

Cada texto se transforma en un vector. La idea es que textos con significado parecido tengan vectores cercanos.

In [10]:
# Ten en cuenta que los modelos Transformer no requieren de lematización, quitar stopwords, ni tokenización previa. De hecho, estos procesos pueden eliminar información relevante para el modelo.
embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

embeddings = embedding_model.encode(
    documents,
    show_progress_bar=True
)

print("Shape de embeddings:", embeddings.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Shape de embeddings: (10000, 384)


## Cambiar el modelo de embeddings

`SentenceTransformers` permite utilizar modelos preentrenados descargados desde Hugging Face:

https://huggingface.co/models

Para usar otro modelo, consulta la documentación del mismo.
```

### Importante

Distintos modelos pueden producir resultados muy diferentes.

Algunas diferencias habituales:

- algunos modelos son más rápidos que otros,
- algunos funcionan mejor en español,
- algunos generan embeddings más precisos pero consumen más memoria.

Cambiar el modelo puede modificar completamente:

- la `cosine similarity`,
- el clustering,
- los topics detectados por `BERTopic`,
- el `coherence score`.

Por tanto, la elección del modelo de embeddings influye directamente en la representación semántica del corpus y en la calidad final del análisis.

## 5. Cosine similarity

La similitud coseno mide si dos vectores apuntan en una dirección parecida.

Aquí elegimos un documento y buscamos los documentos semánticamente más similares.

In [11]:
query_idx = 0

similarities = cosine_similarity(
    [embeddings[query_idx]],
    embeddings
)[0]

# Excluimos el propio documento con [1:6]
top_similar = similarities.argsort()[::-1][1:6]

print("DOCUMENTO ORIGINAL")
print("=" * 80)
print(f"Índice: {query_idx}")
print(f"Categoría: {df.iloc[query_idx][GROUP_COLUMN]}")
print(documents[query_idx])

print("\nDOCUMENTOS MÁS SIMILARES")
for idx in top_similar:
    print("\n" + "=" * 80)
    print(f"Índice: {idx}")
    print(f"Similitud coseno: {similarities[idx]:.3f}")
    print(f"Categoría: {df.iloc[idx][GROUP_COLUMN]}")
    print(documents[idx])

DOCUMENTO ORIGINAL
Índice: 0
Categoría: eldiario.es
En la competición de visualizaciones del vídeo más impactante del Día de Aragón, en esta ocasión no ganó ni el del Gobierno de Aragón ni el de la empresa cervecera de la tierra. Ganaron contundentemente las deplorables imágenes de El Alcoraz proyectando en todo el mundo, pocas horas después de las celebraciones del 23 de abril, una imagen macarra de los dos principales clubes de fútbol de la comunidad autónoma. Mal juego, victoria de penalti del Huesca, pelea y agresiones al final del partido, tres expulsiones y, a falta de 5 partidos para finalizar la temporada, los dos equipos en puestos de descenso a la tercera categoría del fútbol profesional.

Era el partido de rivalidad autonómica, el mejor cierre festivo a la programación de San Jorge, pero no asistieron ni el presidente de Aragón, todavía en funciones, ni las alcaldesas de Zaragoza y de Huesca. Todo un síntoma de los malos tiempos que corren para los equipos de fútbol más repr

## 6. Matriz de similitud coseno

También podemos calcular la similitud entre todos los documentos y visualizarla como una matriz.

In [12]:
similarity_matrix = cosine_similarity(embeddings)

sim_df = pd.DataFrame(
    similarity_matrix,
    index=[f"doc_{i}" for i in range(len(documents))],
    columns=[f"doc_{i}" for i in range(len(documents))]
)

display(sim_df.round(2))

,doc_0,doc_1,doc_2,doc_3,doc_4,doc_5,doc_6,doc_7,doc_8,doc_9,...,doc_9990,doc_9991,doc_9992,doc_9993,doc_9994,doc_9995,doc_9996,doc_9997,doc_9998,doc_9999
doc_0,1.00,0.11,1.00,0.44,0.15,0.25,0.01,0.25,0.11,0.20,...,0.25,0.35,0.16,0.23,0.34,0.33,0.21,0.40,0.40,0.26
doc_1,0.11,1.00,0.11,0.29,0.25,0.26,0.26,0.37,0.30,0.17,...,0.36,0.30,0.21,0.13,0.23,0.19,0.25,0.27,0.05,0.21
doc_2,1.00,0.11,1.00,0.44,0.15,0.25,0.01,0.25,0.11,0.20,...,0.25,0.35,0.16,0.23,0.34,0.33,0.21,0.40,0.40,0.26
doc_3,0.44,0.29,0.44,1.00,0.25,0.22,0.11,0.35,0.24,0.10,...,0.48,0.38,0.20,0.08,0.49,0.22,0.19,0.35,0.32,0.29
doc_4,0.15,0.25,0.15,0.25,1.00,0.09,0.26,0.21,0.10,0.32,...,0.32,0.09,0.07,0.05,0.09,0.09,0.04,0.07,0.00,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
doc_9995,0.33,0.19,0.33,0.22,0.09,0.13,0.18,0.11,0.32,0.51,...,0.23,0.49,0.29,0.32,0.32,1.00,0.44,0.54,0.30,0.24
doc_9996,0.21,0.25,0.21,0.19,0.04,0.09,0.14,0.24,0.50,0.28,...,0.25,0.57,0.35,0.26,0.36,0.44,1.00,0.47,0.37,0.21
doc_9997,0.40,0.27,0.40,0.35,0.07,0.23,0.13,0.24,0.25,0.37,...,0.35,0.64,0.46,0.35,0.48,0.54,0.47,1.00,0.49,0.24
doc_9998,0.40,0.05,0.40,0.32,0.00,0.35,0.05,0.30,0.22,0.25,...,0.23,0.47,0.34,0.37,0.49,0.30,0.37,0.49,1.00,0.29


![BERTopic](BERTopic.png)

## 7. Topic modeling semántico con BERTopic

BERTopic no parte solo de frecuencias de palabras. Primero usa embeddings para representar los documentos semánticamente y después agrupa documentos similares.

In [13]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    language="multilingual",  
    min_topic_size=2,  # número mínimo de documentos por tema
    verbose=True
)

topics, probs = topic_model.fit_transform(
    documents,
    embeddings
)

df["topic"] = topics

display(df[[TEXT_COLUMN, GROUP_COLUMN, "topic"]])

2026-05-07 13:49:08,015 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-07 13:49:41,203 - BERTopic - Dimensionality - Completed ✓
2026-05-07 13:49:41,215 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-07 13:49:42,305 - BERTopic - Cluster - Completed ✓
2026-05-07 13:49:42,323 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-07 13:49:50,863 - BERTopic - Representation - Completed ✓


,texto,periodico,topic
0,"En la competición de visualizaciones del vídeo más impactante del Día de Aragón, en esta ocasión no ganó ni el del Gobierno de Aragón ni el de la empresa cervecera de la tierra. Ganaron contundent...",eldiario.es,30
1,"Hay asuntos que una no abandona al salir del juzgado. Casos que se quedan adheridos a la piel, que modifican la manera de mirar el mundo y que convierten determinados detalles un sonido, una pala...",eldiario.es,866
2,"En la competición de visualizaciones del vídeo más impactante del Día de Aragón, en esta ocasión no ganó ni el del Gobierno de Aragón ni el de la empresa cervecera de la tierra. Ganaron contundent...",eldiario.es,30
3,"Como un gélido bisturí explorador de las miserias humanas definió Miguel Ángel Palomo la película 'Sexo, mentiras y cintas de vídeo', la aclamada y ochentera ópera prima de Steven Soderbergh . E...",eldiario.es,271
4,Un minuto y diez segundos. Eso es lo que dura un vídeo de informativos en televisión. La duración estándar. En ese tiempo quienes trabajamos en ese medio hemos de ser capaces de contar una vida; e...,eldiario.es,354
...,...,...,...
9995,"abascal y sánchez, en una sesión de control. el sondeo de sociométrica que hoy publica el español dibuja un escenario de emergencia para el bloque gubernamental. el psoe, siguiendo la inercia de l...",el español,139
9996,"los argumentos del actual portavoz paradisputar el liderazgo a la mujera la que, hace sólo tres meses, rendía exagerada pleitesía como su “más leal escudero” son tan inconsistentes que merecen poc...",el español,-1
9997,"pedro sánchez y oriol junqueras, este jueves en moncloa. para tratar de recomponer la mayoría de investidura perdida e insuflar vida a esta legislatura agonizante,pedro sánchezse ha lanzado a reco...",el español,586
9998,"quiso el destino que el último día de campaña de las primarias del pp coincidiera con mi visita al castillo dechenonceau, construido a modo de deslumbrante puente sobre elloiray entregado, como pr...",el español,218


![KeyBERTInspired](KeyBertInspired.png)

[BERTopic](https://maartengr.github.io/BERTopic/algorithm/algorithm.html) permite realizar una implementación más detallada. En el siguiente código se muestra una implementación más avanzada.


In [14]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction import text

stopwords_es = [
    'de', 'la', 'que', 'el', 'en', 'y', 'a', 'los', 'del', 'se',
    'las', 'por', 'un', 'para', 'con', 'no', 'una', 'su', 'al',
    'lo', 'es', 'si', 'ha', 'más', 'pero', 'sus', 'le', 'ya',
    'o', 'este', 'sí', 'porque', 'esta', 'entre', 'cuando',
    'muy', 'sin', 'sobre', 'también', 'me', 'hasta', 'hay',
    'donde', 'quien', 'desde', 'todo', 'nos', 'durante',
    'todos', 'uno', 'les', 'ni', 'contra', 'otros', 'ese',
    'eso', 'ante', 'ellos', 'e', 'esto', 'mí', 'antes',
    'algunos', 'qué', 'unos', 'yo', 'otro', 'otras', 'otra',
    'él', 'tanto', 'esa', 'estos', 'mucho', 'quienes', 'nada',
    'muchos', 'cual', 'poco', 'ella', 'estar', 'estas',
    'algunas', 'algo', 'nosotros', 'como', 'ser', 'son', 'está', 
    'han', 'sino', 'solo'
]

mis_stopwords = list(set(stopwords_es))

In [15]:
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer

#  Reduce dimensionality
umap_model = UMAP(
    n_neighbors=7,   # Número de vecinos para construir el grafo de proximidad en el espacio de alta dimensión. Un valor más alto puede capturar mejor la estructura global, mientras que un valor más bajo se enfoca en la estructura local.
    n_components=5,   # Número de dimensiones a las que se reducirá el espacio de embeddings. Un valor más bajo puede facilitar la visualización y el clustering, pero también puede perder información relevante.
    min_dist=0.0,      
    metric="cosine",  
    random_state=42
)

# Cluster reduced embeddings
hdbscan_model = HDBSCAN(
    min_cluster_size=200,   # Número mínimo de documentos por tema. Un valor más bajo puede generar más temas, pero también puede incluir temas menos coherentes.
    min_samples=7,        # Controla lo estricto que es HDBSCAN para considerar que una zona es realmente un cluster (en función de la densidad de muestras). Un valor más bajo puede generar más clusters, pero también puede incluir clusters menos coherentes.
    metric="euclidean",   # Métrica de distancia para calcular la similitud entre documentos
    cluster_selection_method="eom", # Método de selección de clusters "excess of mass" que prioriza clusters densos y bien definidos.
    prediction_data=True
)

# Tokenize topics
vectorizer_model = CountVectorizer(
    stop_words=mis_stopwords,
    min_df=2,
    ngram_range=(1, 2)
)

# Create topic representation
ctfidf_model = ClassTfidfTransformer()

# All steps together
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    language="multilingual",
    verbose=True
)

topics, probs = topic_model.fit_transform(
    documents,
    embeddings
)

df["topic"] = topics

display(df[[TEXT_COLUMN, GROUP_COLUMN, "topic"]])

2026-05-07 13:50:04,169 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


2026-05-07 13:50:19,157 - BERTopic - Dimensionality - Completed ✓
2026-05-07 13:50:19,158 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-07 13:50:19,543 - BERTopic - Cluster - Completed ✓
2026-05-07 13:50:19,547 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-07 13:50:36,695 - BERTopic - Representation - Completed ✓


,texto,periodico,topic
0,"En la competición de visualizaciones del vídeo más impactante del Día de Aragón, en esta ocasión no ganó ni el del Gobierno de Aragón ni el de la empresa cervecera de la tierra. Ganaron contundent...",eldiario.es,0
1,"Hay asuntos que una no abandona al salir del juzgado. Casos que se quedan adheridos a la piel, que modifican la manera de mirar el mundo y que convierten determinados detalles un sonido, una pala...",eldiario.es,0
2,"En la competición de visualizaciones del vídeo más impactante del Día de Aragón, en esta ocasión no ganó ni el del Gobierno de Aragón ni el de la empresa cervecera de la tierra. Ganaron contundent...",eldiario.es,0
3,"Como un gélido bisturí explorador de las miserias humanas definió Miguel Ángel Palomo la película 'Sexo, mentiras y cintas de vídeo', la aclamada y ochentera ópera prima de Steven Soderbergh . E...",eldiario.es,0
4,Un minuto y diez segundos. Eso es lo que dura un vídeo de informativos en televisión. La duración estándar. En ese tiempo quienes trabajamos en ese medio hemos de ser capaces de contar una vida; e...,eldiario.es,0
...,...,...,...
9995,"abascal y sánchez, en una sesión de control. el sondeo de sociométrica que hoy publica el español dibuja un escenario de emergencia para el bloque gubernamental. el psoe, siguiendo la inercia de l...",el español,0
9996,"los argumentos del actual portavoz paradisputar el liderazgo a la mujera la que, hace sólo tres meses, rendía exagerada pleitesía como su “más leal escudero” son tan inconsistentes que merecen poc...",el español,0
9997,"pedro sánchez y oriol junqueras, este jueves en moncloa. para tratar de recomponer la mayoría de investidura perdida e insuflar vida a esta legislatura agonizante,pedro sánchezse ha lanzado a reco...",el español,0
9998,"quiso el destino que el último día de campaña de las primarias del pp coincidiera con mi visita al castillo dechenonceau, construido a modo de deslumbrante puente sobre elloiray entregado, como pr...",el español,0


## 8. Información general de topics

In [16]:

topic_info = topic_model.get_topic_info()
display(topic_info)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1477,-1_españa_años_gobierno_fue,"[españa, años, gobierno, fue, vez, puede, política, estado, dos, hace]","[el 22 de octubre de 2009, cuatro mil personas se reunieron en el palacio de los deportes para celebrar los primeros 20 años deel mundo. hasta sus mayores detractores reconocían entonces que la hi..."
1,0,4552,0_sánchez_gobierno_años_españa,"[sánchez, gobierno, años, españa, política, dos, vez, era, puede, pp]","[adaptemos al mundo de hoy la cita más vigente de los dramas históricos deshakespeare: ""en la corona hueca que ciñe las sienes de un ególatra indispensable tiene su corte la muerte"". lo del “ególa..."
2,1,1646,1_españa_personas_años_gobierno,"[españa, personas, años, gobierno, vivienda, millones, social, puede, animales, cada]","[un obrero de la construcción.sams solutions, unsplash. si hablamos de listas de espera en nuestro sistema nacional de salud no descubrimos nada nuevo. a estas alturas, todo el mundo sabe que las ..."
3,2,1442,2_trump_estados_unidos_estados unidos,"[trump, estados, unidos, estados unidos, guerra, venezuela, china, mundo, rusia, eeuu]","[Juan Torres López • Opinión • 22/05/2025 Los economistas de todo el mundo, los políticos e incluso la gente normal que analiza lo que está ocurriendo en Estados Unidos, desde que Donald Trump asu..."
4,3,403,3_israel_gaza_palestina_genocidio,"[israel, gaza, palestina, genocidio, palestino, israelí, pueblo, palestinos, estado, internacional]","[Santiago González Vallejo • Opinión • 22/09/2025 La misión del barco Handala, de la Flotilla de la Libertad , consiste en romper el ilegal bloqueo de la palestina Gaza, ejecutado por parte de Isr..."
5,4,273,4_irán_israel_guerra_trump,"[irán, israel, guerra, trump, estados, unidos, estados unidos, iraní, nuclear, régimen]","[Enrique Muñoz Gamarra • Opinión • 13/04/2025 Donald Trump que, en los debates electorales de finales de 2024 gustaba envolverse con la aureola: “Mi esperanza es que mi mayor legado sea el de un p..."
6,5,207,5_fascismo_mundo_capitalismo_mundial,"[fascismo, mundo, capitalismo, mundial, países, social, fue, china, poder, global]","[Enrique Muñoz Gamarra • Opinión • 24/01/2026 La relación base-superestructura es una de las variables fundamentales del materialismo histórico, básico en los análisis de las sociedades. La base, ..."


## 9. Palabras principales por topic

In [17]:
unique_topics = sorted(set(topics))

for topic_id in unique_topics:
    if topic_id == -1:
        continue

    print("\n" + "=" * 80)
    print(f"TOPIC {topic_id}")

    words = topic_model.get_topic(topic_id)
    topic_words = [word for word, score in words[:10]]

    print(topic_words)


TOPIC 0
['sánchez', 'gobierno', 'años', 'españa', 'política', 'dos', 'vez', 'era', 'puede', 'pp']

TOPIC 1
['españa', 'personas', 'años', 'gobierno', 'vivienda', 'millones', 'social', 'puede', 'animales', 'cada']

TOPIC 2
['trump', 'estados', 'unidos', 'estados unidos', 'guerra', 'venezuela', 'china', 'mundo', 'rusia', 'eeuu']

TOPIC 3
['israel', 'gaza', 'palestina', 'genocidio', 'palestino', 'israelí', 'pueblo', 'palestinos', 'estado', 'internacional']

TOPIC 4
['irán', 'israel', 'guerra', 'trump', 'estados', 'unidos', 'estados unidos', 'iraní', 'nuclear', 'régimen']

TOPIC 5
['fascismo', 'mundo', 'capitalismo', 'mundial', 'países', 'social', 'fue', 'china', 'poder', 'global']


In [18]:
words = topic_model.get_topic(1)
print(words)

[('españa', np.float64(0.010004932333317576)), ('personas', np.float64(0.009880417759629567)), ('años', np.float64(0.00906105060592134)), ('gobierno', np.float64(0.008895201177801956)), ('vivienda', np.float64(0.008278963250138511)), ('millones', np.float64(0.008198629258688013)), ('social', np.float64(0.008154212080415963)), ('puede', np.float64(0.007502057368866033)), ('animales', np.float64(0.007155901798086316)), ('cada', np.float64(0.007109083315801845))]


## 10. Coherence score

El coherence score intenta medir si las palabras principales de cada topic son coherentes entre sí.

La métrica más usada para ello es la $C_v$ ([Exploring the Space of Topic Coherence Measures](https://dl.acm.org/doi/10.1145/2684822.2685324))

#### Cálculo del score coherence global

In [19]:
# Tokenización sencilla para clase.
# Si el proyecto lo requiere, se puede mejorar con limpieza, stopwords o lematización.
tokenized_docs = [doc.lower().split() for doc in documents]
dictionary = Dictionary(tokenized_docs) # Se crea un diccionario de Gensim a partir de los documentos tokenizados, lo que permite mapear cada palabra a un ID único.

topics_for_coherence = []

for topic_id in unique_topics:
    if topic_id == -1:
        continue

    words = topic_model.get_topic(topic_id)
    topic_words = [word for word, score in words[:10]] # Solo las 10 palabras más representativas del topic
    topics_for_coherence.append(topic_words)

if len(topics_for_coherence) > 0:
    coherence_model = CoherenceModel(
        topics=topics_for_coherence, # Las listas de palabras representativas de cada topic se pasan como input para calcular la coherencia.
        texts=tokenized_docs, # Los textos tokenizados se pasan para calcular la coherencia basada en la co-ocurrencia de palabras en los documentos.
        dictionary=dictionary, # El diccionario de Gensim se pasa para mapear las palabras a sus IDs y calcular la coherencia correctamente.
        coherence="c_v" # La medida de coherencia "c_v" es una de las más utilizadas y combina la co-ocurrencia de palabras con la distancia semántica entre ellas.
    )

    coherence = coherence_model.get_coherence()
    # Cálculo de la coherencia global del modelo de topics utilizando la medida "c_v"
    print(f"Coherence Score: {coherence:.4f}")  # Un valor más alto indica que las palabras del topic tienden a aparecer juntas en los documentos, lo que sugiere un tema más coherente.
else:
    print("No hay topics suficientes para calcular coherence score.")

Coherence Score: 0.5884


#### Cálculo del score coherence por tema

In [20]:
#print de los coherence scores de los topics
tokenized_docs = [doc.lower().split() for doc in documents]

# Diccionario de Gensim:
# palabra -> ID único
dictionary = Dictionary(tokenized_docs)

print("=" * 80)
print("COHERENCE SCORE POR TOPIC")
print("=" * 80)

for topic_id in unique_topics:

    # Ignorar outliers de BERTopic
    if topic_id == -1:
        continue

    # Obtener palabras representativas del topic
    words = topic_model.get_topic(topic_id)

    # Seleccionar las 10 palabras más importantes
    topic_words = [word for word, score in words[:10]]

    # El coherence model espera una lista de topics,
    # aunque aquí solo evaluemos uno
    coherence_model = CoherenceModel(
        topics=[topic_words],
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence="c_v"
    )

    # Calcular coherence del topic individual
    coherence = coherence_model.get_coherence()

    print(f"Topic {topic_id}: {coherence:.4f}")

COHERENCE SCORE POR TOPIC
Topic 0: 0.3908
Topic 1: 0.3584
Topic 2: 0.7237
Topic 3: 0.8136
Topic 4: 0.7554
Topic 5: 0.4888


In [21]:
dictionary

## 11. Cruce topic × variable de interés

Este bloque permite responder preguntas como:

- ¿Qué topics aparecen más en cada periódico?
- ¿Qué topics se asocian a mejores o peores valoraciones?
- ¿Qué topics aparecen más en cada género musical?
- ¿Qué topics están sobrerrepresentados en noticias clickbait?

In [22]:
if GROUP_COLUMN in df.columns:
    topic_by_group = (
        df.groupby(GROUP_COLUMN)["topic"]
        .value_counts(normalize=True)
        .rename("proportion")
        .reset_index()
        .sort_values([GROUP_COLUMN, "proportion"], ascending=[True, False])
    )

    display(topic_by_group)
else:
    print(f"La columna {GROUP_COLUMN} no existe en el dataframe.")

,periodico,topic,proportion
0,Público,0,0.4810
1,Público,-1,0.2010
2,Público,1,0.1240
3,Público,2,0.0960
4,Público,3,0.0475
5,Público,4,0.0260
6,Público,5,0.0245
7,Tercera Información,2,0.3750
8,Tercera Información,0,0.1890
9,Tercera Información,-1,0.1380


## 12. Documentos representativos por topic

Para interpretar un topic no basta con mirar las palabras principales. Conviene leer varios documentos reales asignados a ese topic.

In [23]:
for topic_id in unique_topics:
    if topic_id == -1:
        continue

    print("\n" + "=" * 80)
    print(f"TOPIC {topic_id}")

    topic_docs = df[df["topic"] == topic_id]

    for i, (_, row) in enumerate(topic_docs.head(3).iterrows(), start=1):
        print(f"\nDocumento {i} | {GROUP_COLUMN}: {row.get(GROUP_COLUMN, 'NA')}")
        print(row[TEXT_COLUMN][:500])


TOPIC 0

Documento 1 | periodico: eldiario.es
En la competición de visualizaciones del vídeo más impactante del Día de Aragón, en esta ocasión no ganó ni el del Gobierno de Aragón ni el de la empresa cervecera de la tierra. Ganaron contundentemente las deplorables imágenes de El Alcoraz proyectando en todo el mundo, pocas horas después de las celebraciones del 23 de abril, una imagen macarra de los dos principales clubes de fútbol de la comunidad autónoma. Mal juego, victoria de penalti del Huesca, pelea y agresiones al final del partido, t

Documento 2 | periodico: eldiario.es
Hay asuntos que una no abandona al salir del juzgado. Casos que se quedan adheridos a la piel, que modifican la manera de mirar el mundo y que convierten determinados detalles un sonido, una palabra, un número en algo imposible de olvidar. Como abogada, he aprendido que hay procedimientos que se estudian, otros que se sufren y algunos que directamente te acompañan para siempre. El juicio por presuntos delitos

## 13. Visualizaciones de BERTopic

Estas visualizaciones suelen funcionar mejor con datasets más grandes (con dos cluster no podrás visualizar)

In [24]:
!pip install nbformat
!pip install notebook ipykernel jupyter

In [25]:

topic_model.visualize_topics()
topic_model.visualize_barchart()
topic_model.visualize_heatmap()
